# Building a RAG Pipeline

## From raw text to answerable questions

Over the past four notebooks, we have built every piece of a modern search pipeline:

1. **Text cleaning** — stripping HTML, invisible characters, OCR artefacts
2. **Deduplication** — removing redundant copies with hashing and similarity
3. **Chunking** — breaking long documents into searchable pieces
4. **Vector search** — finding relevant chunks by meaning, not just keywords

Now we wire them together into a **Retrieval-Augmented Generation** pipeline — the architecture behind ChatGPT with browsing, Bing AI, Perplexity, and every enterprise "ask your documents" product.

The idea is beautifully simple. When a user asks a question:
1. Search your documents for the most relevant chunks
2. Paste those chunks into a prompt alongside the question
3. Send the prompt to an LLM

The LLM answers using the provided context rather than its training data. It is *grounded* in your documents. That is what "retrieval-augmented" means.

In [ ]:
%pip install -q scikit-learn pyarrow

In [ ]:
import json
import re
import html
import unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## The RAG architecture

Here is the full pipeline, end to end:

```
                        INGESTION (offline, runs once)
                        ==============================
                        
  Raw documents  -->  Clean  -->  Chunk  -->  Embed  -->  Store in index
                        
                        
                        RETRIEVAL (online, per query)
                        =============================
                        
  User question  -->  Embed query  -->  Search index  -->  Top-k chunks
                                                                |
                                                                v
                                                     Construct prompt
                                                     (system + context + question)
                                                                |
                                                                v
                                                       Send to LLM  -->  Answer
```

The ingestion pipeline runs once (or whenever documents change). The retrieval pipeline runs on every user query. We will build both.

## Part 1: The ingestion pipeline

We will take raw British Library texts, clean them, and chunk them into an indexed collection. This is a condensed version of what we built across notebooks 1-3.

### Step 1: Load raw texts

In [ ]:
raw_df = pd.read_csv("../data/bl_digitised_texts_raw.csv")
print(f"Raw texts: {len(raw_df)} documents")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head(3)

In [ ]:
# Let's also load the longer research papers for chunking
with open("../data/bl_research_papers.json") as f:
    papers = json.load(f)

print(f"Research papers: {len(papers)}")
for p in papers:
    print(f"  {p['paper_id']}: {p['title']} ({len(p['text'])} chars)")

### Step 2: Clean

A compact cleaning function that chains the steps from notebook 1.

In [ ]:
def clean_text(text):
    """Clean a raw text: remove HTML, normalise Unicode, strip boilerplate."""
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)
    text = html.unescape(text)
    
    # Unicode normalisation — collapses non-breaking spaces, fancy quotes, etc.
    text = unicodedata.normalize("NFKC", text)
    
    # Remove zero-width characters
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    
    # Remove boilerplate headers/footers
    text = re.sub(r"Page \d+ of \d+.*?Collection", "", text)
    
    # Collapse multiple whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [ ]:
# Clean all short texts
raw_df["clean_text"] = raw_df["raw_text"].apply(clean_text)

# Clean the research papers
for paper in papers:
    paper["clean_text"] = clean_text(paper["text"])

# Quick before/after comparison
sample = raw_df.iloc[0]
print("BEFORE:")
print(repr(sample["raw_text"][:200]))
print("\nAFTER:")
print(repr(sample["clean_text"][:200]))

### Step 3: Chunk

Short texts from the digitised collection are already chunk-sized. The research papers need splitting. We will use paragraph-based chunking with overlap.

In [ ]:
def chunk_text(text, source_id, max_chunk_chars=500, overlap_chars=100):
    """Split text into overlapping chunks."""
    chunks = []
    
    # Try paragraph-based splitting first
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    # If the text has no paragraph breaks, use sentence-based splitting
    if len(paragraphs) <= 1:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) > max_chunk_chars and current_chunk:
                chunks.append(current_chunk.strip())
                # Overlap: keep the last portion
                words = current_chunk.split()
                overlap_words = words[-max(1, len(words) // 4):]
                current_chunk = " ".join(overlap_words) + " " + sentence
            else:
                current_chunk += " " + sentence
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
    else:
        current_chunk = ""
        for para in paragraphs:
            if len(current_chunk) + len(para) > max_chunk_chars and current_chunk:
                chunks.append(current_chunk.strip())
                # Keep overlap from the end of the previous chunk
                current_chunk = current_chunk[-overlap_chars:] + " " + para
            else:
                current_chunk += " " + para if current_chunk else para
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
    
    return [
        {"source_id": source_id, "chunk_index": i, "chunk_text": c}
        for i, c in enumerate(chunks)
    ]

In [ ]:
all_chunks = []

# Chunk the short digitised texts (most are already short enough to be single chunks)
for _, row in raw_df.iterrows():
    if len(row["clean_text"]) > 20:  # Skip near-empty texts
        chunks = chunk_text(row["clean_text"], source_id=row["text_id"])
        all_chunks.extend(chunks)

# Chunk the research papers
for paper in papers:
    chunks = chunk_text(paper["clean_text"], source_id=paper["paper_id"])
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame(all_chunks)
chunks_df["chunk_id"] = [f"C-{i:04d}" for i in range(len(chunks_df))]

print(f"Total chunks: {len(chunks_df)}")
print(f"Average chunk length: {chunks_df['chunk_text'].str.len().mean():.0f} chars")
chunks_df.head()

### Step 4: Embed and index

We use TF-IDF to build embeddings. In production, you would call an embedding API (OpenAI, Cohere, or a self-hosted model) and store the vectors in a vector database (Pinecone, Weaviate, Qdrant, pgvector). The data engineering is identical — only the embedding step changes.

In [ ]:
# Build the TF-IDF index
vectoriser = TfidfVectorizer(max_features=384, stop_words="english")
chunk_vectors = vectoriser.fit_transform(chunks_df["chunk_text"]).toarray().astype(np.float32)

# Normalise to unit vectors
norms = np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
norms[norms == 0] = 1
chunk_vectors = chunk_vectors / norms

print(f"Index built: {chunk_vectors.shape[0]} chunks x {chunk_vectors.shape[1]} dimensions")

That is the ingestion pipeline complete. We now have a searchable index of cleaned, chunked British Library texts. Let us wrap it in a single function.

In [ ]:
def ingest(raw_texts_df, papers_list):
    """Full ingestion pipeline: clean -> chunk -> embed -> index."""
    # Clean
    raw_texts_df = raw_texts_df.copy()
    raw_texts_df["clean_text"] = raw_texts_df["raw_text"].apply(clean_text)
    cleaned_papers = []
    for p in papers_list:
        p = dict(p)
        p["clean_text"] = clean_text(p["text"])
        cleaned_papers.append(p)
    
    # Chunk
    all_chunks = []
    for _, row in raw_texts_df.iterrows():
        if len(row["clean_text"]) > 20:
            all_chunks.extend(chunk_text(row["clean_text"], source_id=row["text_id"]))
    for paper in cleaned_papers:
        all_chunks.extend(chunk_text(paper["clean_text"], source_id=paper["paper_id"]))
    
    chunks = pd.DataFrame(all_chunks)
    chunks["chunk_id"] = [f"C-{i:04d}" for i in range(len(chunks))]
    
    # Embed
    vec = TfidfVectorizer(max_features=384, stop_words="english")
    vectors = vec.fit_transform(chunks["chunk_text"]).toarray().astype(np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1
    vectors = vectors / norms
    
    return chunks, vectors, vec

print("Ingestion function defined.")

## Part 2: The retrieval pipeline

Given a user question, we need to:
1. Embed the query into the same vector space
2. Find the most similar chunks
3. Return them ranked by relevance

In [ ]:
def retrieve(query, vectoriser, chunk_vectors, chunks_df, top_k=3):
    """Retrieve the top-k most relevant chunks for a query."""
    # Embed the query
    query_vec = vectoriser.transform([query]).toarray().astype(np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm > 0:
        query_vec = query_vec / query_norm
    
    # Compute similarities
    similarities = cosine_similarity(query_vec, chunk_vectors)[0]
    
    # Get top-k
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source_id": chunks_df.iloc[idx]["source_id"],
            "score": similarities[idx],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
        })
    
    return results

In [ ]:
# Test retrieval
question = "What was the economic impact of the Black Death on English workers?"
results = retrieve(question, vectoriser, chunk_vectors, chunks_df, top_k=3)

print(f"Question: {question}\n")
for r in results:
    print(f"  [{r['score']:.4f}] {r['source_id']} / {r['chunk_id']}")
    print(f"  {r['chunk_text'][:150]}...\n")

## Part 3: Prompt construction

This is the most important step — and the one most people underestimate.

A RAG prompt has three parts:
1. **System message** — tells the LLM how to behave (use only the provided context, admit when unsure)
2. **Context** — the retrieved chunks, clearly formatted
3. **User question** — what the researcher actually asked

The quality of your prompt determines the quality of the answer. Get this wrong and even the best retrieval is wasted.

In [ ]:
def build_prompt(question, retrieved_chunks):
    """Construct a RAG prompt from a question and retrieved chunks."""
    
    system_message = (
        "You are a research assistant for the British Library. "
        "Answer the researcher's question using ONLY the provided context passages. "
        "If the context does not contain enough information to answer fully, say so clearly. "
        "Cite the source IDs when referencing specific information. "
        "Be concise and precise."
    )
    
    # Format context
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Source: {chunk['source_id']}]\n{chunk['chunk_text']}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = (
        f"SYSTEM: {system_message}\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n\n"
        f"ANSWER:"
    )
    
    return prompt

In [ ]:
# Build and display the prompt
question = "What was the economic impact of the Black Death on English workers?"
results = retrieve(question, vectoriser, chunk_vectors, chunks_df, top_k=3)
prompt = build_prompt(question, results)

print("=" * 70)
print("THE ASSEMBLED PROMPT")
print("=" * 70)
print(prompt)
print("=" * 70)
print(f"\nPrompt length: {len(prompt)} characters")
print(f"Approximate tokens: ~{len(prompt) // 4}")

There it is. The entire pipeline — cleaning, chunking, embedding, retrieval — exists to produce this one string.

Everything we have built across five notebooks converges into a single prompt that an LLM can read and answer. The context is grounded in real documents from the British Library. The LLM does not need to hallucinate or rely on its training data. It reads what we retrieved and answers based on that.

We cannot actually send this to an LLM from the browser (no API keys, no network calls from Pyodide), but in a production system, this prompt would go to an API endpoint and come back with a grounded, cited answer.

### The full pipeline in one function

In [ ]:
def rag_query(question, vectoriser, chunk_vectors, chunks_df, top_k=3):
    """Full RAG pipeline: retrieve -> build prompt."""
    # Retrieve
    results = retrieve(question, vectoriser, chunk_vectors, chunks_df, top_k=top_k)
    
    # Build prompt
    prompt = build_prompt(question, results)
    
    return {
        "question": question,
        "retrieved_chunks": results,
        "prompt": prompt,
        "prompt_chars": len(prompt),
        "approx_tokens": len(prompt) // 4,
    }

In [ ]:
# Try several questions
questions = [
    "How did the Industrial Revolution change working conditions in factories?",
    "What role did Tudor explorers play in expanding English influence?",
    "How was cholera understood and treated in Victorian London?",
]

for q in questions:
    result = rag_query(q, vectoriser, chunk_vectors, chunks_df)
    print(f"\nQ: {q}")
    print(f"   Retrieved {len(result['retrieved_chunks'])} chunks")
    print(f"   Top score: {result['retrieved_chunks'][0]['score']:.4f}")
    print(f"   Prompt: ~{result['approx_tokens']} tokens")
    print(f"   Sources: {[c['source_id'] for c in result['retrieved_chunks']]}")

## Part 4: Hybrid retrieval

In practice, the best search systems combine keyword matching and vector search. Why? Because they have complementary strengths:

- **Keyword search** finds exact matches reliably — if a researcher searches for "ENIAC" they want documents containing that exact term
- **Vector search** finds semantic matches — "early computers" should find documents about Colossus and Bletchley Park even if they never use the word "computer"

Hybrid retrieval combines both scores. A simple approach: normalise each score to [0, 1] and take a weighted average.

In [ ]:
def hybrid_retrieve(query, vectoriser, chunk_vectors, chunks_df, 
                    top_k=5, keyword_weight=0.3, vector_weight=0.7):
    """Hybrid retrieval: combine keyword and vector scores."""
    
    # Keyword scores
    query_terms = query.lower().split()
    keyword_scores = []
    for text in chunks_df["chunk_text"]:
        text_lower = text.lower()
        score = sum(text_lower.count(term) for term in query_terms)
        keyword_scores.append(score)
    keyword_scores = np.array(keyword_scores, dtype=np.float32)
    
    # Normalise keyword scores to [0, 1]
    kw_max = keyword_scores.max()
    if kw_max > 0:
        keyword_scores = keyword_scores / kw_max
    
    # Vector scores
    query_vec = vectoriser.transform([query]).toarray().astype(np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm > 0:
        query_vec = query_vec / query_norm
    vector_scores = cosine_similarity(query_vec, chunk_vectors)[0]
    
    # Combine
    combined_scores = (keyword_weight * keyword_scores) + (vector_weight * vector_scores)
    
    top_indices = np.argsort(combined_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source_id": chunks_df.iloc[idx]["source_id"],
            "keyword_score": keyword_scores[idx],
            "vector_score": vector_scores[idx],
            "combined_score": combined_scores[idx],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
        })
    
    return results

In [ ]:
# Compare pure vector vs hybrid for a query where keywords help
query = "Babbage analytical engine"

print(f"Query: '{query}'\n")

print("--- Vector only ---")
vec_results = retrieve(query, vectoriser, chunk_vectors, chunks_df, top_k=3)
for r in vec_results:
    print(f"  [{r['score']:.4f}] {r['chunk_text'][:100]}...")

print("\n--- Hybrid (30% keyword + 70% vector) ---")
hyb_results = hybrid_retrieve(query, vectoriser, chunk_vectors, chunks_df, top_k=3)
for r in hyb_results:
    print(f"  [kw={r['keyword_score']:.2f} vec={r['vector_score']:.4f} combined={r['combined_score']:.4f}]")
    print(f"  {r['chunk_text'][:100]}...")

The `keyword_weight` and `vector_weight` parameters control the balance. For a domain like the British Library where researchers often search for specific named entities (people, places, artefacts), giving some weight to keywords prevents the vector search from drifting too far from what the user actually typed.

## Part 5: Evaluating retrieval quality

How do you know if your retrieval is any good? In a production system you would build a proper evaluation set: a list of questions paired with known-relevant documents. Here we will do a quick manual evaluation.

In [ ]:
# Define some test queries with expected topics
test_queries = [
    {
        "question": "How did plague affect the feudal system?",
        "expected_topics": ["plague", "black death", "feudal", "labour", "peasant"],
    },
    {
        "question": "What machines were invented during the Industrial Revolution?",
        "expected_topics": ["factory", "machine", "steam", "spinning", "industrial"],
    },
    {
        "question": "How was disease controlled in Victorian cities?",
        "expected_topics": ["cholera", "sanitation", "sewer", "public health", "victorian"],
    },
]

for test in test_queries:
    results = retrieve(test["question"], vectoriser, chunk_vectors, chunks_df, top_k=3)
    
    # Check if retrieved chunks contain expected topic words
    hits = 0
    for r in results:
        text_lower = r["chunk_text"].lower()
        if any(topic in text_lower for topic in test["expected_topics"]):
            hits += 1
    
    precision = hits / len(results) if results else 0
    print(f"Q: {test['question']}")
    print(f"   Relevant: {hits}/{len(results)} (precision: {precision:.0%})")
    print(f"   Top result: {results[0]['chunk_text'][:100]}...\n")

This is a rough evaluation — we are checking for topic words rather than true relevance. A proper evaluation would have human-labelled judgements for each query-document pair. But even this rough check tells us whether the retrieval is broadly on target or completely lost.

The standard metrics for retrieval evaluation are:

- **Precision@k** — of the top k results, what fraction are relevant?
- **Recall@k** — of all relevant documents, what fraction appear in the top k?
- **MRR (Mean Reciprocal Rank)** — how high does the first relevant result appear?

In production, you would track these metrics over time, and any drop would trigger an investigation.

## Part 6: The production gap

We have built a working RAG pipeline in a browser. That is genuinely impressive. But let us be honest about what a production system needs that we do not have.

### What we built vs what production needs

| Component | Our version | Production version |
|-----------|------------|-------------------|
| **Embeddings** | TF-IDF (384-dim, sparse) | Dense model like `all-MiniLM-L6-v2` or OpenAI `text-embedding-3-small` |
| **Vector store** | Numpy array in memory | Vector database (Pinecone, Weaviate, Qdrant, pgvector) |
| **Chunking** | Simple paragraph/sentence split | Semantic chunking, table/image handling |
| **Search** | Brute-force cosine similarity | Approximate nearest neighbour (HNSW, IVF) |
| **LLM** | Prompt printed to screen | API call to GPT-4, Claude, or similar |
| **Scale** | ~2,000 documents | Millions of documents |
| **Updates** | Re-run the whole pipeline | Incremental ingestion, delta processing |
| **Monitoring** | Manual inspection | Retrieval metrics, latency tracking, drift detection |
| **Caching** | None | Query cache, embedding cache |

### The risks

RAG is not magic. Real-world systems face real problems:

- **Hallucination** — the LLM may generate plausible-sounding answers that are not in the context. Good prompts help ("only use the provided context"), but do not eliminate the risk entirely.
- **Retrieval failures** — if the right chunk is not retrieved, the LLM cannot answer correctly no matter how good it is. Retrieval quality is the bottleneck.
- **Copyright and licensing** — the British Library texts may have usage restrictions. Making them searchable via AI raises legal questions.
- **Bias** — the collection reflects historical biases. An AI system trained on colonial-era documents will reflect colonial perspectives unless carefully handled.
- **Cost** — embedding millions of documents and serving thousands of queries is not free. Embedding costs, vector storage costs, and LLM API costs add up.

### The data engineer's role

Notice something about this entire module. We have not trained a model. We have not fine-tuned anything. We have not written a single neural network.

What we have done is:
- Cleaned text
- Deduplicated documents
- Chunked text intelligently
- Built search indices
- Constructed prompts
- Evaluated retrieval quality

This is data engineering. The LLM is a powerful component, but it is useless without clean, well-structured, searchable data feeding into it. The data pipeline is the foundation. Every AI product is only as good as its data pipeline.

That is your job.

---

## Exercises

### Exercise 1: End-to-end RAG

Write a function `ask(question)` that:
1. Retrieves the top 3 chunks using hybrid retrieval
2. Builds a prompt
3. Prints the prompt nicely formatted with clear sections
4. Returns a dict with the question, retrieved sources, and the prompt

Test it on three questions of your choosing about the British Library collection.

In [ ]:
# Your code here


### Exercise 2: Tune the weights

Run the same 5 queries with different hybrid weights:
- 100% keyword, 0% vector
- 50% keyword, 50% vector
- 30% keyword, 70% vector
- 0% keyword, 100% vector

For each configuration, assess the quality of the top result. Which weight combination works best? Does the answer depend on the type of query?

In [ ]:
# Your code here


### Exercise 3: Prompt engineering

Write three different system messages for the prompt:
1. A formal academic assistant
2. A museum tour guide explaining to the public
3. A children's educator making history accessible

For the same question and retrieved chunks, build all three prompts and compare them. How does the system message change the expected answer style?

In [ ]:
# Your code here


### Your turn: Production design

Write a brief design document (in a markdown cell below) for taking this pipeline to production at the British Library. Address:

1. Which embedding model would you use, and why?
2. Which vector database would you choose for 170 million items?
3. How would you handle incremental updates when new texts are digitised?
4. What monitoring would you put in place?
5. How would you handle the copyright and bias risks mentioned above?

There are no right answers here — this is about applying what you have learned to a real-world scenario.

*Write your design document here.*


---

## Summary

In this notebook we built a complete Retrieval-Augmented Generation pipeline:

1. **Ingestion** — clean raw texts, chunk them, compute embeddings, build a searchable index
2. **Retrieval** — embed a query, find the nearest chunks by cosine similarity
3. **Prompt construction** — assemble system message, context, and question into a single prompt
4. **Hybrid search** — combine keyword and vector scores for better retrieval
5. **Evaluation** — measure precision to know if retrieval is working

The assembled prompt is the output of the entire pipeline. Every step — from stripping HTML in notebook 1 to computing cosine similarity in notebook 4 — exists to produce that one string.

This is the architecture behind every modern AI search product. The details vary (better embeddings, faster vector stores, smarter chunking), but the structure is the same. And the data engineering — cleaning, structuring, indexing, monitoring — is where the real work happens.

You now have the foundation to build these systems.